In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Load base reward model
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

# Load example dataset 
data = [
    {"prompt": "What is AI?", "response": "Artificial Intelligence is the simulation of human intelligence by machines.", "rating": 4.5},
    {"prompt": "Tell me a joke", "response": "I'm not funny, sorry!", "rating": 2.0}
]
dataset = Dataset.from_list(data)

# Preprocessing
def preprocess(example):
    example["text"] = example["prompt"] + " " + example["response"]
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(preprocess)
tokenized_dataset = tokenized_dataset.rename_column("rating", "labels")

# Training config
args = TrainingArguments(
    output_dir="./reward_model",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_dir="./logs",
    save_strategy="epoch"
)

# Train
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)
trainer.train()
model.save_pretrained("./reward_model")


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gadep\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a

Step,Training Loss


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [2]:
def generate_with_reward(prompt, model, tokenizer, reward_model, reward_tokenizer):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    output_ids = model.generate(input_ids, max_new_tokens=50)
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    reward_input = reward_tokenizer(prompt + " " + response, return_tensors="pt", truncation=True).to(model.device)
    with torch.no_grad():
        reward = reward_model(**reward_input).logits.item()

    return {"response": response, "reward": reward}


In [4]:
!pip install flask



   ---------------------------------------- 0/4 [werkzeug]
   ------------------------------ --------- 3/4 [flask]
   ---------------------------------------- 4/4 [flask]



In [1]:
from flask import Flask, request, jsonify
import json

app = Flask(__name__)
feedback_file = "feedback.json"

@app.route('/submit_feedback', methods=['POST'])
def submit_feedback():
    data = request.get_json()
    with open(feedback_file, "a") as f:
        f.write(json.dumps(data) + "\n")
    return jsonify({"message": "Feedback saved"})

@app.route('/chat', methods=['POST'])
def chat():
    prompt = request.get_json()["prompt"]
    result = generate_with_reward(prompt, model, tokenizer, reward_model, reward_tokenizer)
    return jsonify(result)

if __name__ == "__main__":
    app.run(port=5001)


 * Tip: There are .env files present. Install python-dotenv to use them.


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
